# Retail Demand Forecasting

## 2. Data Preprocessing

This stage prepares the raw datasets for exploratory analysis and forecasting model development.

The preprocessing workflow includes:

- Data type validation
- Missing value assessment
- Duplicate detection
- Date consistency checks
- Categorical value validation
- Integration of store-level information
- Integration of holiday and event information
- Integration of external economic variables
- Validation of the resulting analytical dataset

The objective is to produce a consistent and modelling-ready dataset while preserving the temporal structure required for forecasting.

In [33]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [34]:
DATA_DIR = "data"

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
stores = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))
oil = pd.read_csv(os.path.join(DATA_DIR, "oil.csv"))
holidays = pd.read_csv(os.path.join(DATA_DIR, "holidays_events.csv"))
transactions = pd.read_csv(os.path.join(DATA_DIR, "transactions.csv"))

In [35]:
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])
oil["date"] = pd.to_datetime(oil["date"])
holidays["date"] = pd.to_datetime(holidays["date"])
transactions["date"] = pd.to_datetime(transactions["date"])

## 2.1 Sales Data Validation

The historical sales dataset is validated before integration with external information. The checks focus on missing observations, duplicate records, invalid values, and the uniqueness of store-product-date combinations.

In [36]:
sales_quality = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Unique Dates",
        "Unique Stores",
        "Unique Product Families",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Value": [
        len(train),
        train.shape[1],
        train["date"].nunique(),
        train["store_nbr"].nunique(),
        train["family"].nunique(),
        train.isna().sum().sum(),
        train.duplicated().sum()
    ]
})

sales_quality

,Metric,Value
0,Rows,3000888
1,Columns,6
2,Unique Dates,1684
3,Unique Stores,54
4,Unique Product Families,33
5,Missing Values,0
6,Duplicate Rows,0


In [37]:
target_validation = pd.DataFrame({
    "Metric": [
        "Minimum Sales",
        "Maximum Sales",
        "Mean Sales",
        "Median Sales",
        "Zero Sales Observations",
        "Negative Sales Observations"
    ],
    "Value": [
        train["sales"].min(),
        train["sales"].max(),
        train["sales"].mean(),
        train["sales"].median(),
        (train["sales"] == 0).sum(),
        (train["sales"] < 0).sum()
    ]
})

target_validation

,Metric,Value
0,Minimum Sales,0.00
1,Maximum Sales,"124,717.00"
2,Mean Sales,357.78
3,Median Sales,11.00
4,Zero Sales Observations,"939,130.00"
5,Negative Sales Observations,0.00


In [38]:
date_range = pd.date_range(
    train["date"].min(),
    train["date"].max(),
    freq="D"
)

missing_dates = date_range.difference(
    train["date"].drop_duplicates()
)

print("Expected dates:", len(date_range))
print("Observed dates:", train["date"].nunique())
print("Missing dates:", len(missing_dates))

Expected dates: 1688
Observed dates: 1684
Missing dates: 4


In [39]:
missing_dates[:20]

DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'], dtype='datetime64[ns]', freq=None)

In [40]:
duplicate_keys = train.duplicated(
    subset=["date", "store_nbr", "family"]
).sum()

print("Duplicate date-store-family combinations:", duplicate_keys)

Duplicate date-store-family combinations: 0


In [41]:
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [42]:
stores["store_nbr"].nunique()

54

In [43]:
stores["type"].value_counts()

type
D    18
C    15
A     9
B     8
E     4
Name: count, dtype: int64

In [44]:
stores["city"].nunique(), stores["state"].nunique()

(22, 16)

In [45]:
oil.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [46]:
print("Oil data:", oil["date"].min(), "to", oil["date"].max())

Oil data: 2013-01-01 00:00:00 to 2017-08-31 00:00:00


In [47]:
holidays.head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [48]:
holidays["type"].value_counts()

type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

In [49]:
holidays["locale"].value_counts()

locale
National    174
Local       152
Regional     24
Name: count, dtype: int64

In [50]:
transactions.head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [51]:
transactions["store_nbr"].nunique()

54

In [52]:
train_model = train.merge(
    stores,
    on="store_nbr",
    how="left",
    validate="many_to_one"
)
print("Original rows:", len(train))
print("After store merge:", len(train_model))

Original rows: 3000888
After store merge: 3000888


In [53]:
train_model = train_model.merge(
    oil,
    on="date",
    how="left"
)
train_model.shape
train_model.isna().sum()

id                  0
date                0
store_nbr           0
family              0
sales               0
onpromotion         0
city                0
state               0
type                0
cluster             0
dcoilwtico     928422
dtype: int64

In [54]:
train_model = train_model.merge(
    transactions,
    on=["date", "store_nbr"],
    how="left"
)

In [55]:
print("Rows after transactions merge:", len(train_model))

Rows after transactions merge: 3000888


In [56]:
holiday_dates = (
    holidays.loc[
        holidays["transferred"] == False,
        ["date"]
    ]
    .drop_duplicates()
    .assign(is_holiday=1)
)

In [57]:
train_model = train_model.merge(
    holiday_dates,
    on="date",
    how="left"
)

In [58]:
train_model["is_holiday"] = (
    train_model["is_holiday"]
    .fillna(0)
    .astype(int)
)

## 2.2 Integrated Dataset Validation

Following the integration of store, economic, transaction, and holiday information, the resulting dataset is validated to ensure that the original observation structure has been preserved and that no unintended row duplication has occurred.

In [59]:
final_validation = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Unique Dates",
        "Unique Stores",
        "Unique Product Families",
        "Duplicate Rows",
        "Missing Values"
    ],
    "Value": [
        len(train_model),
        train_model.shape[1],
        train_model["date"].nunique(),
        train_model["store_nbr"].nunique(),
        train_model["family"].nunique(),
        train_model.duplicated().sum(),
        train_model.isna().sum().sum()
    ]
})

final_validation

,Metric,Value
0,Rows,3000888
1,Columns,13
2,Unique Dates,1684
3,Unique Stores,54
4,Unique Product Families,33
5,Duplicate Rows,0
6,Missing Values,1174206


In [60]:
train_model.columns.tolist()

['id',
 'date',
 'store_nbr',
 'family',
 'sales',
 'onpromotion',
 'city',
 'state',
 'type',
 'cluster',
 'dcoilwtico',
 'transactions',
 'is_holiday']

In [61]:
OUTPUT_PATH = "data/processed_sales_data.csv"

train_model.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Processed dataset exported successfully: {OUTPUT_PATH}")

Processed dataset exported successfully: data/processed_sales_data.csv
